In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
#  directory to save trained models
os.makedirs("models", exist_ok=True)


DROWSINESS_DIR = "/kaggle/input/datasets/ismailnasri20/driver-drowsiness-dataset-ddd/Driver Drowsiness Dataset (DDD)"
UTKFACE_DIR = "/kaggle/input/datasets/jangedoo/utkface-new/UTKFace"

IMG_SIZE = (64, 64)

In [ ]:
def load_sleep_data(data_dir):
    images, labels = [], []
    classes = ['Non Drowsy', 'Drowsy'] # 0: Awake, 1: Asleep
    
    for label_idx, cls in enumerate(classes):
        path = os.path.join(data_dir, cls)
       
        image_paths = glob.glob(os.path.join(path, "*.[jJ][pP][gG]")) + \
                      glob.glob(os.path.join(path, "*.[pP][nN][gG]"))
        
        for img_path in image_paths:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                images.append(img)
                labels.append(label_idx)
                
    images = np.array(images, dtype='float32') / 255.0  # Normalize [0, 1]
    labels = np.array(labels)
    return images, labels

print("Loading Sleep Detection Dataset from:", DROWSINESS_DIR)
X_sleep, y_sleep = load_sleep_data(DROWSINESS_DIR)

In [ ]:
# Dataset Balancing 
awake_idx = np.where(y_sleep == 0)[0]
asleep_idx = np.where(y_sleep == 1)[0]
min_samples = min(len(awake_idx), len(asleep_idx))

np.random.seed(42)
balanced_awake = np.random.choice(awake_idx, min_samples, replace=False)
balanced_asleep = np.random.choice(asleep_idx, min_samples, replace=False)

balanced_indices = np.concatenate([balanced_awake, balanced_asleep])
X_sleep_bal = X_sleep[balanced_indices]
y_sleep_bal = y_sleep[balanced_indices]

In [ ]:
# Visualize Balanced Sleep Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y_sleep_bal)
plt.title("Balanced Sleep Distribution (0: Awake, 1: Asleep)")
plt.xticks([0, 1], ['Awake', 'Asleep'])
plt.savefig("models/sleep_class_distribution.png")
plt.show()
plt.close()

In [ ]:
# Split Train/Test Sets
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sleep_bal, y_sleep_bal, test_size=0.2, random_state=42, stratify=y_sleep_bal
)

In [ ]:
# Augmentation & Build CNN
datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1, horizontal_flip=True)
datagen.fit(X_train_s)

In [ ]:
def build_sleep_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
sleep_model = build_sleep_cnn()
history_s = sleep_model.fit(
    datagen.flow(X_train_s, y_train_s, batch_size=32),
    epochs=15,
    validation_data=(X_test_s, y_test_s)
)
sleep_model.save("models/sleep_detector_cnn.h5")

In [ ]:
# Save Accuracy & Loss Curves
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_s.history['accuracy'], label='Train')
plt.plot(history_s.history['val_accuracy'], label='Val')
plt.title('Sleep Model Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_s.history['loss'], label='Train')
plt.plot(history_s.history['val_loss'], label='Val')
plt.title('Sleep Model Loss')
plt.legend()
plt.savefig("models/sleep_metrics.png")
plt.show()
plt.close()

In [ ]:

# Age Estimation Model 
def load_age_data(data_dir, max_samples=5000):
    images, ages = [], []
    image_paths = glob.glob(os.path.join(data_dir, "*.jpg"))
    
    for img_path in image_paths[:max_samples]:
        filename = os.path.basename(img_path)
        parts = filename.split("_")
        if len(parts) >= 4 and parts[0].isdigit():
            age = int(parts[0])
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                images.append(img)
                ages.append(age)
                
    images = np.array(images, dtype='float32') / 255.0
    ages = np.array(ages)
    return images, ages

In [ ]:
print("Loading Age Estimation Dataset from:", UTKFACE_DIR)
X_age, y_age = load_age_data(UTKFACE_DIR)

In [ ]:
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_age, y_age, test_size=0.2, random_state=42
)

In [ ]:
def build_age_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='linear') # Continuous regression output for age
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [ ]:
age_model = build_age_cnn()
history_a = age_model.fit(
    X_train_a, y_train_a,
    batch_size=32,
    epochs=15,
    validation_data=(X_test_a, y_test_a)
)
age_model.save("models/age_estimator_cnn.h5")



In [ ]:
plt.figure(figsize=(12, 4))

#  Mean Absolute Error 
plt.subplot(1, 2, 1)
plt.plot(history_a.history['mae'], label='Train MAE', color='blue')
plt.plot(history_a.history['val_mae'], label='Val MAE', color='orange')
plt.title('Age Model Accuracy (Mean Absolute Error)')
plt.xlabel('Epochs')
plt.ylabel('MAE (Years Error)')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

#  Loss Curve 
plt.subplot(1, 2, 2)
plt.plot(history_a.history['loss'], label='Train Loss', color='blue')
plt.plot(history_a.history['val_loss'], label='Val Loss', color='orange')
plt.title('Age Model Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("models/age_metrics.png")
plt.show()
plt.close()